# Video upload & inspection

Building block for video-based analysis shared across demos:

* capillary-breakup experiments
* particle-tracking microrheology
* differential dynamic microscopy

> ⚠️ **Kernel requirement.** Decoding video needs OpenCV (`cv2`), which has no Pyodide
> build, so this notebook cannot run in the JupyterLite in-browser kernel — use a full
> Python kernel (local Jupyter, Colab, Binder). In JupyterLite the file uploader is also
> subject to the browser upload-size limit, so keep test clips small.

The notebook uploads (or downloads) a test clip, inspects its metadata and frames, and
computes a simple frame-difference motion summary as a smoke test of the pipeline.

In [ ]:
%pip install -q opencv-python-headless pillow ipywidgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display

upload = widgets.FileUpload(accept='.mp4', multiple=False,
                            description='Upload test mp4')
display(upload)


In [ ]:
# Use the uploaded clip when present, otherwise fetch the repo's test clip (~16 MB).
import urllib.request
from pathlib import Path

TEST_MP4_URL = ("https://raw.githubusercontent.com/rheopy/rheolite/main/"
                "content/video/test.mp4")

files = getattr(globals().get('upload'), 'value', None) or ()
if len(files):
    video_path = files[0]['name']
    Path(video_path).write_bytes(files[0]['content'])
    print(f"using uploaded video: {video_path}")
else:
    video_path = 'test.mp4'
    if not Path(video_path).exists():
        print(f"downloading {TEST_MP4_URL} ...")
        urllib.request.urlretrieve(TEST_MP4_URL, video_path)
    print(f"using repo test clip: {video_path} ({Path(video_path).stat().st_size/1e6:.1f} MB)")


In [ ]:
import cv2

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"could not open {video_path}")
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"{video_path}: {n_frames} frames, {fps:.1f} fps, {width}x{height}, "
      f"duration {n_frames/fps:.1f} s")
cap.release()


In [ ]:
# First / middle / last frame thumbnails
import cv2
from PIL import Image

cap = cv2.VideoCapture(video_path)
thumbs = []
for idx in [0, n_frames // 2, n_frames - 1]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if ok:
        im = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        im.thumbnail((320, 320))
        thumbs.append((idx, im))
cap.release()

for idx, im in thumbs:
    print(f"frame {idx}")
    display(im)


In [ ]:
# Frame-difference motion summary: mean absolute difference between sampled frames.
# A flat line means a static scene; spikes flag motion/events in the clip.
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt

STEP = max(1, n_frames // 60)  # ~60 samples across the clip
cap = cv2.VideoCapture(video_path)
prev, diffs, times = None, [], []
for idx in range(0, n_frames, STEP):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if not ok:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(float)
    if prev is not None:
        diffs.append(np.abs(gray - prev).mean())
        times.append(idx / fps)
    prev = gray
cap.release()

fig, ax = plt.subplots()
ax.plot(times, diffs, '.-', ms=4)
ax.set_xlabel('time [s]')
ax.set_ylabel('mean |frame difference| [counts]')
ax.set_title('motion summary')
fig.tight_layout()
plt.show()
print(f"mean frame difference: {np.mean(diffs):.2f} counts")


## Interactive playback (optional)

Run the cell below on a live kernel to flip through frames with a slider.
It is defined but not executed here so headless runs stay quiet.

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display
import cv2
from PIL import Image

def play_video(path, max_width=480):
    """Frame slider for a video file (interactive use)."""
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    def _frame_png(i):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, frame = cap.read()
        im = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        im.thumbnail((max_width, max_width))
        buf = io.BytesIO()
        im.rotate(270, expand=True).save(buf, format='PNG')  # chronos camera is mounted sideways
        return buf.getvalue()

    img_w = widgets.Image(value=_frame_png(0), format='png')
    slider = widgets.IntSlider(min=0, max=n - 1, value=0, description='Frame:',
                               continuous_update=False)
    slider.observe(lambda ch: setattr(img_w, 'value', _frame_png(ch['new'])), names='value')
    return widgets.VBox([slider, img_w])

# play_video(video_path)  # <-- uncomment for interactive playback
print("play_video() defined — uncomment the last line to use it interactively")
